In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report
import warnings
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)


df.columns = df.columns.str.strip()

non_numeric_cols = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
df.drop(columns=non_numeric_cols, inplace=True)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

X = df.drop('Label', axis=1)
y = df['Label']

X_train_sup, X_test, y_train_sup, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

X_train_unsup = X_train_sup[y_train_sup == 0]

scaler = StandardScaler()
X_train_sup_scaled = scaler.fit_transform(X_train_sup)
X_test_scaled = scaler.transform(X_test)
X_train_unsup_scaled = scaler.transform(X_train_unsup)

warnings.filterwarnings('ignore')
results = {}

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_sup_scaled, y_train_sup)
y_pred_rf = rf_model.predict(X_test_scaled)
results['Random Forest'] = classification_report(y_test, y_pred_rf, target_names=['Normal (0)', 'DDoS (1)'])

ocsvm_model = OneClassSVM(kernel='rbf', gamma='auto', nu=0.01)
ocsvm_model.fit(X_train_unsup_scaled)

y_pred_ocsvm = ocsvm_model.predict(X_test_scaled)

y_pred_ocsvm_final = np.where(y_pred_ocsvm == 1, 0, 1)
results['One-Class SVM'] = classification_report(y_test, y_pred_ocsvm_final, target_names=['Normal (0)', 'DDoS (1)'])

if_model = IsolationForest(contamination='auto', random_state=42, n_jobs=-1)
if_model.fit(X_train_unsup_scaled)

y_pred_if = if_model.predict(X_test_scaled)

y_pred_if_final = np.where(y_pred_if == 1, 0, 1)
results['Isolation Forest'] = classification_report(y_test, y_pred_if_final, target_names=['Normal (0)', 'DDoS (1)'])

print("\n" + "="*50)
print("  MODEL PERFORMANS RAPORLARI")
print("="*50)

for model_name, report in results.items():
    print(f"\nModel: {model_name}")
    print("-" * 30)
    print(report)
    print("-" * 30)

Saving Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv to Friday-WorkingHours-Afternoon-DDos.pcap_ISCX (1).csv

  MODEL PERFORMANS RAPORLARI

Model: Random Forest
------------------------------
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00     29306
    DDoS (1)       1.00      1.00      1.00     38408

    accuracy                           1.00     67714
   macro avg       1.00      1.00      1.00     67714
weighted avg       1.00      1.00      1.00     67714

------------------------------

Model: One-Class SVM
------------------------------
              precision    recall  f1-score   support

  Normal (0)       0.53      0.99      0.69     29306
    DDoS (1)       0.97      0.33      0.49     38408

    accuracy                           0.61     67714
   macro avg       0.75      0.66      0.59     67714
weighted avg       0.78      0.61      0.58     67714

------------------------------

Model: Isolation Forest
-----------

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score
import warnings

file_name = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
df = pd.read_csv(file_name)

df.columns = df.columns.str.strip()
non_numeric_cols = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
df.drop(columns=non_numeric_cols, inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

X = df.drop('Label', axis=1)
y = df['Label']

X_train_sup, X_test, y_train_sup, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train_unsup = X_train_sup[y_train_sup == 0]

scaler = StandardScaler()
X_train_sup_scaled = scaler.fit_transform(X_train_sup)
X_test_scaled = scaler.transform(X_test)
X_train_unsup_scaled = scaler.transform(X_train_unsup)

warnings.filterwarnings('ignore')
results = {}
anomaly_scores = {}

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_model.fit(X_train_sup_scaled, y_train_sup)
y_pred_xgb = xgb_model.predict(X_test_scaled)
results['XGBoost'] = classification_report(y_test, y_pred_xgb, target_names=['Normal (0)', 'DDoS (1)'])

ocsvm_model = OneClassSVM(kernel='rbf', gamma='auto', nu=0.01)
ocsvm_model.fit(X_train_unsup_scaled)
y_pred_ocsvm = ocsvm_model.predict(X_test_scaled)
y_pred_ocsvm_final = np.where(y_pred_ocsvm == 1, 0, 1)
results['One-Class SVM'] = classification_report(y_test, y_pred_ocsvm_final, target_names=['Normal (0)', 'DDoS (1)'])

anomaly_scores['OCSVM_Score'] = -ocsvm_model.decision_function(X_test_scaled)

if_model = IsolationForest(contamination='auto', random_state=42, n_jobs=-1)
if_model.fit(X_train_unsup_scaled)
y_pred_if = if_model.predict(X_test_scaled)
y_pred_if_final = np.where(y_pred_if == 1, 0, 1)
results['Isolation Forest'] = classification_report(y_test, y_pred_if_final, target_names=['Normal (0)', 'DDoS (1)'])

anomaly_scores['IsolationForest_Score'] = -if_model.score_samples(X_test_scaled)

print("\n" + "="*50)
print("  MODEL PERFORMANS RAPORLARI")
print("="*50)

for model_name, report in results.items():
    print(f"\nModel: {model_name}")
    print("-" * 30)
    print(report)
    print("-" * 30)

print("\n" + "="*50)
print("  ANOMALİ SKORLARI (EN YÜKSEK SKORLAR EN ANORMAL OLANLARDIR)")
print("="*50)

X_test_original = X_test.copy()
X_test_original['Actual_Label'] = y_test

for score_name, scores in anomaly_scores.items():

    X_test_original[score_name] = scores

top_anomalies_if = X_test_original.sort_values(by='IsolationForest_Score', ascending=False).head(5)
top_anomalies_ocsvm = X_test_original.sort_values(by='OCSVM_Score', ascending=False).head(5)

print("\n--- Isolation Forest'a Göre En Anormal 5 Kayıt (Yüksek Skor = Anomali) ---")
print(top_anomalies_if[['Actual_Label', 'IsolationForest_Score']].head())

print("\n--- One-Class SVM'e Göre En Anormal 5 Kayıt (Yüksek Skor = Anomali) ---")
print(top_anomalies_ocsvm[['Actual_Label', 'OCSVM_Score']].head())


  MODEL PERFORMANS RAPORLARI

Model: XGBoost
------------------------------
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00      6030
    DDoS (1)       1.00      1.00      1.00      3090

    accuracy                           1.00      9120
   macro avg       1.00      1.00      1.00      9120
weighted avg       1.00      1.00      1.00      9120

------------------------------

Model: One-Class SVM
------------------------------
              precision    recall  f1-score   support

  Normal (0)       0.79      0.98      0.87      6030
    DDoS (1)       0.92      0.49      0.64      3090

    accuracy                           0.81      9120
   macro avg       0.86      0.73      0.76      9120
weighted avg       0.83      0.81      0.79      9120

------------------------------

Model: Isolation Forest
------------------------------
              precision    recall  f1-score   support

  Normal (0)       0.82      0.91      0.86

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report
import warnings

file_name = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
df = pd.read_csv(file_name)

df.columns = df.columns.str.strip()
non_numeric_cols = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
df.drop(columns=non_numeric_cols, inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

X = df.drop('Label', axis=1)
y = df['Label']

X_train_sup, X_test, y_train_sup, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train_unsup = X_train_sup[y_train_sup == 0]

scaler = StandardScaler()
X_train_sup_scaled = scaler.fit_transform(X_train_sup)
X_test_scaled = scaler.transform(X_test)
X_train_unsup_scaled = scaler.transform(X_train_unsup)

warnings.filterwarnings('ignore')
results = {}
anomaly_scores = {}

mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=10, random_state=42)
mlp_model.fit(X_train_sup_scaled, y_train_sup)
y_pred_mlp = mlp_model.predict(X_test_scaled)
results['MLP Classifier'] = classification_report(y_test, y_pred_mlp, target_names=['Normal (0)', 'DDoS (1)'])

ocsvm_model = OneClassSVM(kernel='rbf', gamma='auto', nu=0.01)
ocsvm_model.fit(X_train_unsup_scaled)
y_pred_ocsvm = ocsvm_model.predict(X_test_scaled)
y_pred_ocsvm_final = np.where(y_pred_ocsvm == 1, 0, 1)
results['One-Class SVM'] = classification_report(y_test, y_pred_ocsvm_final, target_names=['Normal (0)', 'DDoS (1)'])

anomaly_scores['OCSVM_Score'] = -ocsvm_model.decision_function(X_test_scaled)

if_model = IsolationForest(contamination='auto', random_state=42, n_jobs=-1)
if_model.fit(X_train_unsup_scaled)
y_pred_if = if_model.predict(X_test_scaled)
y_pred_if_final = np.where(y_pred_if == 1, 0, 1)
results['Isolation Forest'] = classification_report(y_test, y_pred_if_final, target_names=['Normal (0)', 'DDoS (1)'])

anomaly_scores['IsolationForest_Score'] = -if_model.score_samples(X_test_scaled)

print("\n" + "="*50)
print("  MODEL PERFORMANS RAPORLARI")
print("="*50)

for model_name, report in results.items():
    print(f"\nModel: {model_name}")
    print("-" * 30)
    print(report)
    print("-" * 30)

print("\n" + "="*50)
print("  ANOMALİ SKORLARI (EN YÜKSEK SKORLAR EN ANORMAL OLANLARDIR)")
print("="*50)

X_test_original = X_test.copy()
X_test_original['Actual_Label'] = y_test

for score_name, scores in anomaly_scores.items():
    X_test_original[score_name] = scores

top_anomalies_if = X_test_original.sort_values(by='IsolationForest_Score', ascending=False).head(5)
top_anomalies_ocsvm = X_test_original.sort_values(by='OCSVM_Score', ascending=False).head(5)

print("\n--- Isolation Forest'a Göre En Anormal 5 Kayıt ---")
print(top_anomalies_if[['Actual_Label', 'IsolationForest_Score']].head())

print("\n--- One-Class SVM'e Göre En Anormal 5 Kayıt ---")
print(top_anomalies_ocsvm[['Actual_Label', 'OCSVM_Score']].head())


  MODEL PERFORMANS RAPORLARI

Model: MLP Classifier
------------------------------
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00      7744
    DDoS (1)       1.00      1.00      1.00      7160

    accuracy                           1.00     14904
   macro avg       1.00      1.00      1.00     14904
weighted avg       1.00      1.00      1.00     14904

------------------------------

Model: One-Class SVM
------------------------------
              precision    recall  f1-score   support

  Normal (0)       0.65      0.98      0.79      7744
    DDoS (1)       0.95      0.44      0.60      7160

    accuracy                           0.72     14904
   macro avg       0.80      0.71      0.69     14904
weighted avg       0.80      0.72      0.70     14904

------------------------------

Model: Isolation Forest
------------------------------
              precision    recall  f1-score   support

  Normal (0)       0.73      0.87   

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report
import warnings

file_name = "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
df = pd.read_csv(file_name)

df.columns = df.columns.str.strip()
non_numeric_cols = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
df.drop(columns=non_numeric_cols, inplace=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

le = LabelEncoder()
df['Label'] = le.fit_transform(df['Label'])

X = df.drop('Label', axis=1)
y = df['Label']

X_train_sup, X_test, y_train_sup, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_train_normal = X_train_sup[y_train_sup == 0]


scaler = StandardScaler()
scaler.fit(X_train_sup)
X_test_scaled = scaler.transform(X_test)
X_train_normal_scaled = scaler.transform(X_train_normal)

warnings.filterwarnings('ignore')
results = {}
anomaly_scores = {}

if_model = IsolationForest(contamination='auto', random_state=42, n_jobs=-1)
if_model.fit(X_train_normal_scaled)
y_pred_if = if_model.predict(X_test_scaled)
y_pred_if_final = np.where(y_pred_if == 1, 0, 1)
results['Isolation Forest'] = classification_report(y_test, y_pred_if_final, target_names=['Normal (0)', 'DDoS (1)'])
anomaly_scores['IsolationForest_Score'] = -if_model.score_samples(X_test_scaled)

ocsvm_model = OneClassSVM(kernel='rbf', gamma='auto', nu=0.01)
ocsvm_model.fit(X_train_normal_scaled)
y_pred_ocsvm = ocsvm_model.predict(X_test_scaled)
y_pred_ocsvm_final = np.where(y_pred_ocsvm == 1, 0, 1)
results['One-Class SVM'] = classification_report(y_test, y_pred_ocsvm_final, target_names=['Normal (0)', 'DDoS (1)'])
anomaly_scores['OCSVM_Score'] = -ocsvm_model.decision_function(X_test_scaled)

lof_model = LocalOutlierFactor(n_neighbors=20, novelty=True, contamination='auto', n_jobs=-1)
lof_model.fit(X_train_normal_scaled)
y_pred_lof = lof_model.predict(X_test_scaled)
y_pred_lof_final = np.where(y_pred_lof == 1, 0, 1)
results['Local Outlier Factor'] = classification_report(y_test, y_pred_lof_final, target_names=['Normal (0)', 'DDoS (1)'])
anomaly_scores['LOF_Score'] = -lof_model.decision_function(X_test_scaled)

print("\n" + "="*50)
print("  MODEL PERFORMANS RAPORLARI (Gözetimsiz/Yarı Gözetimsiz)")
print("="*50)

for model_name, report in results.items():
    print(f"\nModel: {model_name}")
    print("-" * 30)
    print(report)
    print("-" * 30)

print("\n" + "="*50)
print("  ANOMALİ SKORLARI: EN ŞÜPHELİ KAYITLAR")
print("  (En Yüksek Skor = En Anormal)")
print("="*50)

X_test_original = X_test.copy()
X_test_original['Actual_Label'] = y_test

for score_name, scores in anomaly_scores.items():
    X_test_original[score_name] = scores

models = ['IsolationForest_Score', 'OCSVM_Score', 'LOF_Score']

for model_score in models:
    top_anomalies = X_test_original.sort_values(by=model_score, ascending=False).head(3)

    print(f"\n--- {model_score}'a Göre En Anormal 3 Kayıt ---")
    print(top_anomalies[['Actual_Label', model_score]].head())


  MODEL PERFORMANS RAPORLARI (Gözetimsiz/Yarı Gözetimsiz)

Model: Isolation Forest
------------------------------
              precision    recall  f1-score   support

  Normal (0)       0.61      0.87      0.72     12319
    DDoS (1)       0.88      0.64      0.74     19153

    accuracy                           0.73     31472
   macro avg       0.75      0.76      0.73     31472
weighted avg       0.78      0.73      0.73     31472

------------------------------

Model: One-Class SVM
------------------------------
              precision    recall  f1-score   support

  Normal (0)       0.50      0.98      0.66     12319
    DDoS (1)       0.97      0.37      0.54     19153

    accuracy                           0.61     31472
   macro avg       0.74      0.68      0.60     31472
weighted avg       0.79      0.61      0.59     31472

------------------------------

Model: Local Outlier Factor
------------------------------
              precision    recall  f1-score   support

 